# Continuous ODE Surrogate for OMWU
This notebook demonstrates how to optimize game matrices to maximize Continuous Regret using `torchdiffeq.odeint_adjoint`.

In [ ]:
import sys
sys.path.append("..")

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torchdiffeq import odeint_adjoint as odeint
from src.dynamics.continuous import OMWUContinuous

plt.style.use('ggplot')

# Use CUDA if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)


In [ ]:
# Hyperparameters
A1, A2 = 2, 2
eta = 0.05
N_steps = 200

from src.engine.ode_optimizer import ODEAdjointOptimizer

# Initialize and run optimizer
optimizer = ODEAdjointOptimizer(A1=A1, A2=A2, eta=eta, N_steps=N_steps, projection_mode="tanh")
final_U1, final_U2, states, t = optimizer.optimize(epochs=50)

print("\nOptimized U1:\n", final_U1.cpu().numpy())
print("Optimized U2:\n", final_U2.cpu().numpy())


In [ ]:
# ==========================================
# Validation against Discrete Engine
# ==========================================
from src.config.schemas import ExperimentConfig, GameConfig, DynamicConfig, ExecutionConfig
from src.engine.runner import ExperimentRunner
from src.engine.statistics import load_experiment_stats

# Create a config using the optimized matrices
config = ExperimentConfig(
    name="optimized_eval",
    game=GameConfig(generator="custom", payoffs=[final_U1.cpu().numpy().tolist(), final_U2.cpu().numpy().tolist()]),
    dynamic=DynamicConfig(algorithm="omwu", strict_theory_eta=False),
    execution=ExecutionConfig(total_steps=N_steps, device="auto", compile=False)
)

# Run discrete engine natively
runner = ExperimentRunner(config=config)
summary = runner.run(target_steps=N_steps)

# Extract native discrete regret
stats_data = load_experiment_stats(output_dir=summary["output_dir"], session_id=summary["session_id"])
steps = stats_data["steps"].numpy()
discrete_regret_1 = stats_data["cum_regrets"].numpy()[:, 0]

# Extract continuous regret trajectory for Player 1
idx = A1 + A2
Z1_traj = states[:, idx : idx+A1]; idx += A1
P1_traj = states[:, idx : idx+1]
continuous_regret_1 = (Z1_traj.max(dim=1).values - P1_traj.squeeze(1)) / eta

import numpy as np
plt.figure(figsize=(10, 6))

# ODE points map to 1..N_steps
plt.plot(np.arange(1, N_steps+1), continuous_regret_1.detach().cpu().numpy(), label="ODE Continuous Regret", linewidth=3, alpha=0.8)
# Discrete points map to logged step indices
plt.plot(steps, discrete_regret_1, label="Discrete Native Regret", linestyle='--', linewidth=2, color='black')

plt.title("Optimized Game: Continuous vs Discrete Regret (Player 1)")
plt.xlabel("Step k")
plt.ylabel("Cumulative Regret")
plt.legend()
plt.tight_layout()
plt.show()
